# Model Prediction - INX Future Inc Employee Performance

## Objective
Use the trained model to make predictions on new employee data.
This notebook demonstrates how to:
1. Load the trained model
2. Prepare new data for prediction
3. Make performance predictions
4. Interpret results for hiring decisions

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

## 1. Load Trained Model and Metadata

In [ ]:
# Load the trained model
model_path = '../../data/processed/best_model.pkl'
model = joblib.load(model_path)

print("Model loaded successfully!")
print(f"Model type: {type(model).__name__}")

# Load model metadata
with open('../../data/processed/model_metadata.json', 'r') as f:
    metadata = json.load(f)

print(f"\nModel Information:")
print(f"  • Model Name: {metadata['model_name']}")
print(f"  • Test Accuracy: {metadata['test_accuracy']:.4f}")
print(f"  • Target Variable: {metadata['target_variable']}")
print(f"  • Number of Features: {len(metadata['features_used'])}")

## 2. Load Test Data for Demonstration

In [ ]:
# Load the processed dataset
df = pd.read_csv('../../data/processed/employee_data_processed.csv')

# Get a sample for demonstration (simulating new candidates)
sample_size = 10
sample_data = df.sample(n=sample_size, random_state=42)

# Extract features (remove target variable)
target = metadata['target_variable']
X_new = sample_data.drop(columns=[target, 'EmpNumber'] if 'EmpNumber' in sample_data.columns else [target])
y_actual = sample_data[target]  # For comparison purposes

print(f"Sample data for prediction: {sample_size} employees")
print(f"\nFeatures shape: {X_new.shape}")
display(X_new.head())

## 3. Make Predictions

In [ ]:
# Make predictions
predictions = model.predict(X_new)
prediction_probabilities = model.predict_proba(X_new)

print("Predictions completed!")
print(f"\nPredicted Performance Ratings:")
print(predictions)

# Get class labels
classes = model.classes_
print(f"\nPerformance Rating Classes: {classes}")

## 4. Create Prediction Results DataFrame

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({
    'Employee_ID': sample_data.index if 'EmpNumber' not in sample_data.columns else sample_data['EmpNumber'].values,
    'Actual_Rating': y_actual.values,
    'Predicted_Rating': predictions
})

# Add probability columns
for i, class_label in enumerate(classes):
    results_df[f'Probability_Rating_{class_label}'] = prediction_probabilities[:, i]

# Add prediction confidence (max probability)
results_df['Confidence'] = prediction_probabilities.max(axis=1)

# Add match indicator
results_df['Prediction_Match'] = (results_df['Actual_Rating'] == results_df['Predicted_Rating']).map({True: '✓', False: '✗'})

print("\n" + "="*100)
print(" " * 35 + "PREDICTION RESULTS")
print("="*100)
display(results_df)

# Calculate accuracy on this sample
sample_accuracy = (results_df['Actual_Rating'] == results_df['Predicted_Rating']).mean()
print(f"\nSample Accuracy: {sample_accuracy:.2%}")

## 5. Prediction for New Candidates (Example)

In [ ]:
# Example: Create a synthetic candidate profile
# This demonstrates how to use the model for hiring decisions

print("\n" + "="*100)
print(" " * 30 + "NEW CANDIDATE PREDICTION EXAMPLE")
print("="*100)

# Use the first row as a template
new_candidate = X_new.iloc[0:1].copy()

print("\nCandidate Profile:")
display(new_candidate.T)

# Make prediction
candidate_prediction = model.predict(new_candidate)[0]
candidate_probabilities = model.predict_proba(new_candidate)[0]

print(f"\n{'='*100}")
print("\nPREDICTION RESULTS:")
print(f"  • Predicted Performance Rating: {candidate_prediction}")
print(f"  • Confidence: {candidate_probabilities.max():.2%}")
print("\nProbability Distribution:")
for class_label, prob in zip(classes, candidate_probabilities):
    print(f"  • Rating {class_label}: {prob:.2%}")

# Hiring recommendation
if candidate_prediction >= 3:
    recommendation = "RECOMMENDED for hire (High performance expected)"
elif candidate_prediction >= 2:
    recommendation = "CONDITIONAL (Average performance expected)"
else:
    recommendation = "NOT RECOMMENDED (Below average performance expected)"

print(f"\nHIRING RECOMMENDATION: {recommendation}")
print(f"{'='*100}")

## 6. Batch Prediction Function

In [ ]:
def predict_employee_performance(model, employee_data, threshold_high=3, threshold_low=2):
    """
    Predict employee performance and provide hiring recommendations.
    
    Parameters:
    -----------
    model : trained sklearn model
        The trained prediction model
    employee_data : pd.DataFrame
        Employee features for prediction
    threshold_high : int
        Threshold for high performance rating
    threshold_low : int
        Threshold for acceptable performance rating
    
    Returns:
    --------
    pd.DataFrame : Predictions with recommendations
    """
    # Make predictions
    predictions = model.predict(employee_data)
    probabilities = model.predict_proba(employee_data)
    confidence = probabilities.max(axis=1)
    
    # Create results
    results = pd.DataFrame({
        'Predicted_Rating': predictions,
        'Confidence': confidence
    })
    
    # Add recommendation
    def get_recommendation(rating):
        if rating >= threshold_high:
            return 'RECOMMENDED'
        elif rating >= threshold_low:
            return 'CONDITIONAL'
        else:
            return 'NOT RECOMMENDED'
    
    results['Hiring_Recommendation'] = results['Predicted_Rating'].apply(get_recommendation)
    
    return results

# Test the function
batch_results = predict_employee_performance(model, X_new)

print("\n" + "="*100)
print(" " * 30 + "BATCH PREDICTION WITH RECOMMENDATIONS")
print("="*100)
display(batch_results)

print("\nRecommendation Summary:")
print(batch_results['Hiring_Recommendation'].value_counts())

## 7. Save Predictions

In [ ]:
# Save prediction results
output_path = '../../data/processed/prediction_results.csv'
results_df.to_csv(output_path, index=False)

print(f"Prediction results saved to: {output_path}")
print("\nPrediction process completed successfully!")

## 8. Model Usage Instructions

In [ ]:
print("\n" + "="*100)
print(" " * 30 + "MODEL USAGE INSTRUCTIONS")
print("="*100)

print("""
TO USE THIS MODEL FOR HIRING DECISIONS:

1. PREPARE NEW CANDIDATE DATA:
   - Ensure all required features are present
   - Features must match the training data format
   - Required features: {}

2. LOAD THE MODEL:
   ```python
   import joblib
   model = joblib.load('../../data/processed/best_model.pkl')
   ```

3. MAKE PREDICTIONS:
   ```python
   predictions = model.predict(candidate_features)
   probabilities = model.predict_proba(candidate_features)
   ```

4. INTERPRET RESULTS:
   - Rating 4: Excellent performer - Strongly RECOMMENDED
   - Rating 3: Good performer - RECOMMENDED
   - Rating 2: Average performer - CONDITIONAL
   - Rating 1: Below average - NOT RECOMMENDED

5. CONFIDENCE LEVELS:
   - High Confidence: > 70%
   - Medium Confidence: 50-70%
   - Low Confidence: < 50% (recommend additional assessment)

""".format(', '.join(metadata['features_used'][:10]) + '...'))

print("="*100)